In [1]:
# pip install -U qwen-asr
# Imports
import librosa
import torch
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from qwen_asr import Qwen3ASRModel

# Dispositivo = GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

Ejecutando en: cuda:0


In [2]:
# Cargar modelo
repo_id = "Qwen/Qwen3-ASR-1.7B"
print(f"Cargando modelo: {repo_id}")

model = Qwen3ASRModel.from_pretrained(
    repo_id,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    max_new_tokens=4096,
)

Cargando modelo: Qwen/Qwen3-ASR-1.7B


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [3]:
audios = []

lst_path = r"..\..\Europarl-ST\fr\en\test\Europarl-ST.fr.en.test.lst"

with open(lst_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        if linea.strip():
            audios.append(str(linea).strip())
        
print(f"Total de audios a procesar: {len(audios)}")

Total de audios a procesar: 166


In [4]:
hypotheses = []
references = []

for audio in tqdm(audios, desc="Transcribiendo con Qwen3-ASR (francés)"):
    
    ref_path = r"..\..\Europarl-ST\fr\en\test\%s\transcription.tok" % audio
    with open(ref_path, "r", encoding="utf-8") as referencia:
        references.append(referencia.read())

    audio_path = r"..\..\Europarl-ST\fr\en\test\%s\audio_clip_diarization.m4a" % audio
    audio_array, sr = librosa.load(audio_path, sr=16000)
    
    results = model.transcribe(audio=(audio_array, sr), language="French")
    
    hypotheses.append(results[0].text)

Transcribiendo con Qwen3-ASR (francés):   0%|          | 0/166 [00:00<?, ?it/s]

C:\Users\carru\AppData\Local\Temp\ipykernel_28340\3985270159.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sr = librosa.load(audio_path, sr=16000)
d:\carru\voxtral-tfg\venv_qwen\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
C:\Users\carru\AppData\Local\Temp\ipykernel_28340\3985270159.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sr = librosa.load(audio_path, sr=16000)
d:\carru\voxtral-tfg\venv_qwen\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Setting

In [5]:
# Crear DataFrame y guardar resultados
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references))

data.to_csv("qwen_raw_results_fr.csv", index=False)

data.head()

,hypothesis,reference
0,"Merci, Monsieur le Président. Madame la Vice P...","Monsieur le Président , Madame la Vice-préside..."
1,"Merci, Monsieur le Président. Tout d'abord, un...","Monsieur le Président , tout d ’ abord une que..."
2,Monde mord de faim. Je remercie Madame la Prés...,"Madame la Présidente , Madame la Commissaire ,..."
3,"Monsieur le Président, le conflit russo géorgi...","Monsieur le Président , le conflit russo-géorg..."
4,"Merci, Madame la Présidente, Monsieur le Minis...","Madame la Présidente , Monsieur le Ministre , ..."


In [6]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(text)) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(text)) for text in data["reference"]]

data.to_csv("qwen_resultados_limpios_fr.csv", index=False)

data.head()

,hypothesis,reference,hypothesis_clean,reference_clean
0,"Merci, Monsieur le Président. Madame la Vice P...","Monsieur le Président , Madame la Vice-préside...",merci monsieur le président madame la vice pré...,monsieur le président madame la vice président...
1,"Merci, Monsieur le Président. Tout d'abord, un...","Monsieur le Président , tout d ’ abord une que...",merci monsieur le président tout d abord une q...,monsieur le président tout d abord une questio...
2,Monde mord de faim. Je remercie Madame la Prés...,"Madame la Présidente , Madame la Commissaire ,...",monde mord de faim je remercie madame la prési...,madame la présidente madame la commissaire la ...
3,"Monsieur le Président, le conflit russo géorgi...","Monsieur le Président , le conflit russo-géorg...",monsieur le président le conflit russo géorgie...,monsieur le président le conflit russo géorgie...
4,"Merci, Madame la Présidente, Monsieur le Minis...","Madame la Présidente , Monsieur le Ministre , ...",merci madame la présidente monsieur le ministr...,madame la présidente monsieur le ministre j ét...


In [7]:
# Cálculo del Word Error Rate (WER)
wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))

print(f"WER final de Qwen3-ASR: {wer * 100:.2f} %")

WER final de Qwen3-ASR: 15.06 %


*(Completar tras la ejecución: WER obtenido en francés.)*